# nano Beatrix — the slot read: which part of the phase table do the path heads' learned code read?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_pillar1k.ipynb)

**The question**: The relabel read (`nano_pillar1j.ipynb`) gave chosen path heads a fresh phase table of the same kind. The head holding the lookup lost almost nothing, so it works by exact matching alone. The other path heads lost more than they do on random phases, so they read the table's values as a learned code. A byte's phase is a sum of one table term per byte of its path, the term in slot k belonging to the byte k back. This notebook changes only some slots: the current byte's term (slot 0), the deeper terms (slots 1-15), or the previous byte's term (slot 1). It also changes all of them again, which must reproduce the relabel read exactly. Every exact match is kept in each case. Every read is split by depth and at the repeated paths. Nothing trains.

| models | sets | slots changed | reads |
|---|---|---|---|
| the path in blocks 1-3, seeds 1-6 | no block, each of blocks 1, 2, 3 alone, all three | all, 0, 1-15, 1 | 120 |
| the path in every block, seeds 1-6 | no block, each block alone, all six | all, 0, 1-15, 1 | 192 |

With no block changed each read equals the model's stored text read, and with every slot changed the reads equal the relabel read's records, bit for bit: the notebook checks both before it ships anything.

**The cells**: C1 pins the trainer and holds the decision block; C2 writes the held-out file and checks the twelve models, their stored text reads and the relabel read's records on [AbstractPhil/geolip-rotary-experiments](https://huggingface.co/AbstractPhil/geolip-rotary-experiments); C3 reads each model with each slot group (a model whose four records are already on the Hub is skipped) and ships the records to `bed/reports/group/` (`<run>_group_relabel_k<slots>_depth.json`); C4 renders each block's cost by slot group.

**Before C1**: Runtime > Change runtime type > the G4 card. The notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on, for the records' upload. Run C1, then Runtime > Restart session and run all. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/ninth_head.py`: `relabel_table(k, slots)`, `relabel_slots`; `rotary_scale.py`: `block_read(mode="relabel", slots=...)`; `tests/test_slot_read.py`).


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.99" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 99), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU (the stored reads were taken in bf16 on the card)'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the shipper reads it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the held-out file, the models, the records
REPO = 'AbstractPhil/geolip-rotary-experiments'      # the models, their stored reads and the relabel read's records are here; the records ship here
TAIL = lambda L: f'_L{L}_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
PIECE = {'early': 'nhpn16b123', 'every': 'nhpn16'}     # the name's piece: the path in blocks 1-3's ninth heads; in every block's
SERIES = {'early': 'pillar1c', 'every': 'pillar1b'}    # their series
TAG = lambda a, s: f'alhena-both_s{s}{TAIL(24000).replace("_askany_", "_askany_p75d_")}_cp4_gc48_kh50_va1_pt10b4h0_krt16r218_{PIECE[a]}_ahdigit1_{SERIES[a]}'
SETS = {'early': [[], [1], [2], [3], [1, 2, 3]],       # no block, each path block alone, all three (five reads a slot group)
        'every': None}                                 # the default: no block, each block alone, all six (eight reads a slot group)
SLOTS = {'all': list(range(16)),                       # every slot: the relabel read again (its records the parity anchor)
         'current': [0],                               # the current byte's term (a key's: the byte before it)
         'deeper': list(range(1, 16)),                 # the deeper terms
         'previous': [1]}                              # the previous byte's term
DEPTH = True                                           # every read split by the byte's depth in its unit, crossed with the repeat split
LABEL = {'early': 'the path in blocks 1-3', 'every': 'the path in every block'}
MODELS = [(f'{LABEL[a]}, seed {s}', a, s, TAG(a, s)) for a in ('early', 'every') for s in range(1, 7)]
TEXT_DOCS, TEXT_MB = 128, 16                           # the stored text reads: the trainer's 128 held-out documents in batches of 16
GROUP_DIR = 'bed/reports/group'                        # this notebook's records (series pillar1k) and the relabel read's (pillar1j)
TEXT_DIR = 'bed/reports/text'                          # the stored text reads (the parity anchor)
print('decision block:', len(MODELS), 'models |', {a: (len(v) if v else 'no block, each alone, all') for a, v in SETS.items()}, '| slot groups',
      {k: (f'{v[0]}-{v[-1]}' if len(v) > 1 else v[0]) for k, v in SLOTS.items()}, '| depth', DEPTH, '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')


In [ ]:
# C2 — THE DATA AND THE MODELS: the held-out head (the trainer's 2,048 reserved fineweb-edu rows), and the twelve models with their
# weights, their ninth heads, their stored text reads and the relabel read's records on the Hub.
import json, os, time
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from geolip.nano_beatrix import bed_corpus
from geolip.nano_beatrix import rotary_scale as rs
HELD = bed_corpus.write_heldout(f'{OUT}/bed')
print('held-out head', os.path.basename(HELD), round(os.path.getsize(HELD) / 1e6, 1), 'MB')
api = HfApi(token=HF_TOKEN)
hub = set(api.list_repo_files(REPO))
need = lambda t: (f'bed/weights/{t}/model.safetensors', f'bed/weights/{t}/extra_head.safetensors', f'bed/weights/{t}/scale.json',
                  f'{TEXT_DIR}/{rs.text_name(t)}.json', f'{GROUP_DIR}/{rs.group_name(t, "relabel", True)}.json')
missing = [lab for lab, a, s, t in MODELS if not all(f in hub for f in need(t))]
assert not missing, ('models, their stored reads or the relabel read records are not on the Hub', missing)
mine = lambda t: [f'{GROUP_DIR}/{rs.group_name(t, "relabel", DEPTH, v)}.json' for v in SLOTS.values()]
before = [lab for lab, a, s, t in MODELS if all(f in hub for f in mine(t))]
print(f'the {len(MODELS)} models with their text reads and relabel records on {REPO}: {len(MODELS)} of {len(MODELS)}; read here before: {len(before)}')


In [ ]:
# C3 — THE READS: each model fetched from the Hub and read with each slot group relabelled (rotary_scale.block_read, mode relabel,
# the depth split on), the reads with no block changed checked against the model's stored text read and the every-slot reads against
# the relabel read's records, bit for bit; the records shipped to bed/reports/group/ (a background shipper with retries). A model whose
# four records are on the Hub or in OUT/group/ is not read again.
import shutil
lship = rs.Shipper(REPO) if HF_TOKEN else None
os.makedirs(f'{OUT}/group', exist_ok=True)
todo = [m for m in MODELS if not all(f in hub for f in mine(m[3]))]
fetch = lambda path: json.load(open(hf_hub_download(REPO, path, local_dir=f'{OUT}/hub', token=HF_TOKEN), encoding='utf-8'))
t0 = time.time()
for i, (lab, a, s, tag) in enumerate(todo):
    outs = {k: f'{OUT}/group/{rs.group_name(tag, "relabel", DEPTH, v)}.json' for k, v in SLOTS.items()}
    if not all(os.path.exists(p) for p in outs.values()):   # records made here before a restart: no download, their checks again
        snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/w', token=HF_TOKEN)
    recs = {k: rs.block_read(f'{OUT}/w/bed/weights/{tag}', outs[k], HELD, sets=SETS[a], n_docs=TEXT_DOCS, mb=TEXT_MB, dev='cuda',
                             log=lambda x: None, mode='relabel', depth=DEPTH, slots=v) for k, v in SLOTS.items()}
    tr = fetch(f'{TEXT_DIR}/{rs.text_name(tag)}.json')
    prev = fetch(f'{GROUP_DIR}/{rs.group_name(tag, "relabel", True)}.json')
    by = {k: {tuple(r['off']): r for r in recs[k]['reads']} for k in SLOTS}
    pb = {tuple(r['off']): r for r in prev['reads']}
    exact = (all(by[k][()]['bits'] == tr['bits'] and by[k][()]['by_repeat'] == tr['by_repeat'] and recs[k].get('slots') == SLOTS[k] for k in SLOTS)
             and set(by['all']) == set(pb) and all(by['all'][o]['bits'] == pb[o]['bits'] and by['all'][o]['by_repeat'] == pb[o]['by_repeat']
                                                   and by['all'][o]['by_depth'] == pb[o]['by_depth'] for o in pb))
    if not exact:
        for p in outs.values():
            os.replace(p, p + '.bad')                  # kept for the look, never shipped
        raise AssertionError((lab, 'no block changed is not the stored text read, or every slot is not the relabel read: the instrument first',
                              {k: by[k][()]['bits'] for k in SLOTS}, tr['bits']))
    if lship:
        lship.submit(f'{OUT}/group', GROUP_DIR)
    shutil.rmtree(f'{OUT}/w/bed/weights/{tag}', ignore_errors=True)   # the session's disk holds one model at a time
    el = time.time() - t0
    base = by['all'][()]['bits']
    print(f"{lab}: as trained {base:.4f}; " + '; '.join(f"{'+'.join(map(str, o))}: " + ', '.join(f"{k} {by[k][o]['bits'] - base:+.4f}" for k in SLOTS)
          for o in by['all'] if o) + f" | {i + 1} of {len(todo)} read, {el / 60:.1f} min spent, about {el / (i + 1) * (len(todo) - i - 1) / 60:.1f} min left", flush=True)
if lship:
    print('the records shipped' if lship.drain() else 'the records still retrying in the background')


In [ ]:
# C4 — THE TABLE (any time; the records made here or read from the Hub): for each model and path block, the cost with every slot
# relabelled, with the current byte's term, with the deeper terms and with the previous byte's term.
from IPython.display import Markdown, display

def _rec(local, hub_path):
    if os.path.exists(local):
        return json.load(open(local, encoding='utf-8'))
    try:
        return json.load(open(hf_hub_download(REPO, hub_path, local_dir=f'{OUT}/hub', token=HF_TOKEN), encoding='utf-8'))
    except Exception:
        return None

for a in ('early', 'every'):
    rows, head = [], None
    for lab, a2, s, tag in MODELS:
        if a2 != a:
            continue
        name = {k: rs.group_name(tag, 'relabel', DEPTH, v) for k, v in SLOTS.items()}
        r = {k: _rec(f'{OUT}/group/{n}.json', f'{GROUP_DIR}/{n}.json') for k, n in name.items()}
        if any(v is None for v in r.values()):
            continue
        b = {k: {tuple(x['off']): x['bits'] for x in v['reads']} for k, v in r.items()}
        sets = [o for o in b['all'] if o]
        if head is None:
            head = ['| seed | ' + ' | '.join(f"block{'s' if len(o) > 1 else ''} {'+'.join(map(str, o))}: " + ' / '.join(SLOTS) for o in sets) + ' |',
                    '|---' * (len(sets) + 1) + '|']
        rows.append(f'| {s} | ' + ' | '.join(' / '.join(f'{b[k][o] - b[k][()]:+.4f}' for k in SLOTS) for o in sets) + ' |')
    if rows:
        display(Markdown(f'**{LABEL[a]}** (held-out bits a byte against the model as trained)\n\n' + '\n'.join(head + rows)))
